In [ ]:
!pip install yfinance --quiet

In [ ]:
import yfinance as yf
import pandas as pd

# 13개 보험사 + 5개 시장 인덱스
tickers = {
    # Treatment 6 (ESG 적극)
    'MUV2.DE': 'Munich Re',
    'SREN.SW': 'Swiss Re',
    'HNR1.DE': 'Hannover Re',
    'SCR.PA':  'SCOR',
    'QBE.AX':  'QBE',
    'BEZ.L':   'Beazley',
    # Control 4 (전통)
    'TRV': 'Travelers',
    'CB':  'Chubb',
    'RNR': 'RenaissanceRe',
    'EG':  'Everest Re',
    # Calibration 3 (NZIA 가입→탈퇴)
    '8766.T': 'Tokio Marine',
    '8725.T': 'MS&AD',
    '8630.T': 'Sompo',
    # 시장 benchmark
    '^GSPC':  'S&P 500',
    '^STOXX': 'Stoxx Europe 600',
    '^N225':  'Nikkei 225',
    '^AXJO':  'ASX 200',
    '^FTSE':  'FTSE 100',
}

START = '2017-01-01'
END   = '2024-12-31'

print("=" * 60)
print("V2 백테스트 데이터 다운로드")
print(f"기간: {START} ~ {END}")
print(f"종목: {len(tickers)}개")
print("=" * 60)

all_data = {}
failed = []

for tk, name in tickers.items():
    print(f"  [{tk:10s}] {name:25s} ... ", end='', flush=True)
    try:
        df = yf.download(tk, start=START, end=END, progress=False, auto_adjust=False)
        if df.empty or 'Adj Close' not in df.columns:
            print("FAIL (no data)")
            failed.append((tk, name))
            continue
        s = df['Adj Close']
        if hasattr(s, 'columns'):
            s = s.iloc[:, 0]
        all_data[tk] = s
        print(f"OK ({len(s)} days)")
    except Exception as e:
        print(f"FAIL ({str(e)[:40]})")
        failed.append((tk, name))

print()
print("=" * 60)
print(f"성공: {len(all_data)}개 / 실패: {len(failed)}개")
if failed:
    print("실패 종목:")
    for tk, name in failed:
        print(f"  - {tk} ({name})")

combined = pd.DataFrame(all_data)
combined.index.name = 'Date'

output_file = 'v2_backtest_data.csv'
combined.to_csv(output_file)

print(f"\n저장 완료: {output_file}")
print(f"  rows: {len(combined)} (거래일)")
print(f"  cols: {len(combined.columns)} (종목)")
print("\n처음 3행:")
print(combined.head(3))
print("\n마지막 3행:")
print(combined.tail(3))


V2 백테스트 데이터 다운로드
기간: 2017-01-01 ~ 2024-12-31
종목: 18개
  [MUV2.DE   ] Munich Re                 ... OK (2033 days)
  [SREN.SW   ] Swiss Re                  ... OK (2010 days)
  [HNR1.DE   ] Hannover Re               ... OK (2033 days)
  [SCR.PA    ] SCOR                      ... OK (2048 days)
  [QBE.AX    ] QBE                       ... OK (2023 days)
  [BEZ.L     ] Beazley                   ... OK (2019 days)
  [TRV       ] Travelers                 ... OK (2011 days)
  [CB        ] Chubb                     ... OK (2011 days)
  [RNR       ] RenaissanceRe             ... OK (2011 days)
  [EG        ] Everest Re                ... OK (2011 days)
  [8766.T    ] Tokio Marine              ... OK (1977 days)
  [8725.T    ] MS&AD                     ... OK (1977 days)
  [8630.T    ] Sompo                     ... OK (1977 days)
  [^GSPC     ] S&P 500                   ... OK (2011 days)
  [^STOXX    ] Stoxx Europe 600          ... OK (2010 days)
  [^N225     ] Nikkei 225                ... OK

In [ ]:
# CSV 자동 다운로드 (브라우저 다운로드 폴더로)

from google.colab import files

files.download('v2_backtest_data.csv')

print('다운로드 완료. 이 CSV를 Claude에 업로드하세요.')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

다운로드 완료. 이 CSV를 Claude에 업로드하세요.
